# TL;DR — Gaze Classifier
## Building a Cognitive State Detector from Eye-Tracking Data

**Project:** TL;DR browser extension — an AI reading assistant that watches *how* you read, not just what you ask.

**This notebook does three things:**
1. Creates a labelled gaze dataset based on cognitive load research
2. Trains a Decision Tree classifier on that dataset
3. Exports the trained model as JavaScript for in-browser use

---

**Why a Decision Tree?** It runs in under 1ms in the browser, needs no ML library, and every decision it makes can be read aloud. For a pitch, that matters.

**Why synthetic data?** There's no publicly available, labelled gaze dataset for reading cognitive states. Publishing that dataset is the long-term goal. For now, we generate it from the published distributions in the literature.


## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.tree import _tree

np.random.seed(42)
print("Ready.")

## 2. The Features

The classifier receives a **9-number vector** computed every 2.5 seconds from raw WebGazer gaze points.

| # | Feature | What it measures |
|---|---|---|
| 1 | `avg_fixation_ms` | Mean duration the eye rests on a word (ms) |
| 2 | `fixation_std` | Stability — high variance means erratic attention |
| 3 | `regression_rate` | Fraction of eye movements going right → left |
| 4 | `saccade_length` | Mean distance between fixations (px) |
| 5 | `saccade_std` | Consistency of those jumps |
| 6 | `gaze_drift_px` | Vertical drift from the text baseline |
| 7 | `scroll_delta_px` | Total scroll movement in the window |
| 8 | `velocity_mean` | Mean gaze speed (px/s) |
| 9 | `line_reread_count` | How many times the eye returned to a line |

Sources: Rayner (1998), Just & Carpenter (1980), Siegenthaler et al. (2011).


## 3. Build the Dataset

Each cognitive state has a known signature. We sample from those distributions.

The output is saved as **`gaze_dataset.csv`** — the dataset we then train from.


In [ ]:
FEATURES = [
    'avg_fixation_ms', 'fixation_std', 'regression_rate',
    'saccade_length',  'saccade_std',  'gaze_drift_px',
    'scroll_delta_px', 'velocity_mean','line_reread_count'
]
N = 350  # per class

def make_class(n, params, label):
    """Sample from normal distributions, clip to valid range."""
    d = {f: np.clip(np.random.normal(mu, sd, n), lo, hi)
         for f, (mu, sd, lo, hi) in params.items()}
    df = pd.DataFrame(d)
    df['label'] = label
    return df

focused = make_class(N, {
    'avg_fixation_ms':   (230, 50,  100, 340),
    'fixation_std':      (55,  18,  15,  110),
    'regression_rate':   (0.10,0.04, 0,  0.22),
    'saccade_length':    (92,  22,  40,  165),
    'saccade_std':       (28,  9,   8,   65),
    'gaze_drift_px':     (14,  6,   2,   35),
    'scroll_delta_px':   (38,  14,  5,   85),
    'velocity_mean':     (285, 75,  100, 520),
    'line_reread_count': (0.8, 0.5, 0,   2.5),
}, 'focused')

skimming = make_class(N, {
    'avg_fixation_ms':   (108, 25,  55,  175),
    'fixation_std':      (38,  14,  8,   75),
    'regression_rate':   (0.05,0.03, 0,  0.13),
    'saccade_length':    (215, 55,  100, 370),
    'saccade_std':       (75,  22,  30,  140),
    'gaze_drift_px':     (28,  12,  5,   65),
    'scroll_delta_px':   (88,  32,  30,  200),
    'velocity_mean':     (540, 130, 250, 950),
    'line_reread_count': (0.2, 0.2, 0,   0.8),
}, 'skimming')

confused = make_class(N, {
    'avg_fixation_ms':   (490, 100, 280, 760),
    'fixation_std':      (125, 42,  50,  260),
    'regression_rate':   (0.36,0.09, 0.15,0.65),
    'saccade_length':    (42,  14,  12,  88),
    'saccade_std':       (18,  7,   4,   42),
    'gaze_drift_px':     (22,  10,  5,   52),
    'scroll_delta_px':   (5,   5,   0,   18),
    'velocity_mean':     (145, 48,  45,  290),
    'line_reread_count': (3.5, 1.2, 1.5, 7.0),
}, 'confused')

zoning_out = make_class(N, {
    'avg_fixation_ms':   (980, 220, 580, 1600),
    'fixation_std':      (215, 65,  80,  420),
    'regression_rate':   (0.18,0.10, 0,  0.48),
    'saccade_length':    (82,  42,  8,   210),
    'saccade_std':       (65,  22,  20,  145),
    'gaze_drift_px':     (60,  22,  22,  130),
    'scroll_delta_px':   (2,   2.5, 0,   8),
    'velocity_mean':     (95,  38,  25,  210),
    'line_reread_count': (0.4, 0.4, 0,   1.5),
}, 'zoning_out')

overloaded = make_class(N, {
    'avg_fixation_ms':   (560, 125, 300, 920),
    'fixation_std':      (160, 52,  60,  320),
    'regression_rate':   (0.44,0.10, 0.25,0.72),
    'saccade_length':    (33,  11,  8,   68),
    'saccade_std':       (14,  5,   3,   32),
    'gaze_drift_px':     (17,  7,   3,   38),
    'scroll_delta_px':   (2,   2.5, 0,   9),
    'velocity_mean':     (115, 38,  38,  245),
    'line_reread_count': (5.2, 1.5, 2.5, 9.5),
}, 'overloaded')

df_raw = pd.concat([focused, skimming, confused, zoning_out, overloaded], ignore_index=True)
df_raw = df_raw.sample(frac=1, random_state=42).reset_index(drop=True)

# --- SAVE THE DATASET FIRST ---
df_raw.to_csv('gaze_dataset.csv', index=False)
print(f"gaze_dataset.csv saved: {len(df_raw)} rows x {len(df_raw.columns)} columns")
print(df_raw['label'].value_counts().to_string())

## 4. Load the Dataset

Now we load it back from disk — exactly as we would with any real CSV dataset.

In [ ]:
# Load from CSV (this is the canonical training step)
df = pd.read_csv('gaze_dataset.csv')
print(f"Loaded: {df.shape}")
print(df.head())

## 5. Feature Distributions

Each state has a distinct visual signature. This is why the classifier can separate them.


In [ ]:
CLASS_ORDER = ['focused','skimming','confused','zoning_out','overloaded']
PAL = {'focused':'#4CAF50','skimming':'#2196F3','confused':'#FF9800','zoning_out':'#9C27B0','overloaded':'#F44336'}

fig, axes = plt.subplots(3, 3, figsize=(15, 10))
for i, feat in enumerate(FEATURES):
    ax = axes.flatten()[i]
    for label in CLASS_ORDER:
        ax.hist(df[df['label']==label][feat], bins=28, alpha=0.5, color=PAL[label], label=label, density=True)
    ax.set_title(feat.replace('_',' ').title(), fontweight='bold', fontsize=10)
    ax.grid(alpha=0.2); ax.set_ylabel('Density')

handles = [mpatches.Patch(color=PAL[l], label=l) for l in CLASS_ORDER]
fig.legend(handles=handles, loc='lower center', ncol=5, fontsize=10, bbox_to_anchor=(0.5,-0.01))
fig.suptitle('Feature Distributions by Cognitive State', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('feature_distributions.png', bbox_inches='tight', dpi=140)
plt.show()

## 6. Class Separability — 2D Scatter Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
pairs = [
    ('avg_fixation_ms','regression_rate','Fixation vs Regression'),
    ('scroll_delta_px','velocity_mean',  'Scroll vs Velocity'),
    ('line_reread_count','saccade_length','Re-reads vs Saccade Length'),
]
for ax, (x, y, title) in zip(axes, pairs):
    for lbl in CLASS_ORDER:
        sub = df[df['label']==lbl]
        ax.scatter(sub[x], sub[y], alpha=0.25, s=12, color=PAL[lbl], label=lbl)
    ax.set_xlabel(x.replace('_',' ')); ax.set_ylabel(y.replace('_',' '))
    ax.set_title(title, fontweight='bold'); ax.legend(fontsize=8); ax.grid(alpha=0.2)
plt.suptitle('Class Separation in Feature Space', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('feature_separability.png', bbox_inches='tight', dpi=140)
plt.show()

## 7. Correlation Heatmap

Correlated features don't hurt a Decision Tree, but they're good to know about.

In [ ]:
plt.figure(figsize=(9,7))
sns.heatmap(df[FEATURES].corr(), annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.4, mask=np.triu(np.ones((9,9),dtype=bool)))
plt.title('Feature Correlation Matrix', fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_matrix.png', bbox_inches='tight', dpi=140)
plt.show()

## 8. Train/Test Split

In [ ]:
X = df[FEATURES].values
y = df['label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

print(f"Train: {len(X_train)} samples")
print(f"Test:  {len(X_test)} samples")

## 9. Train the Decision Tree

**Parameters chosen:**
- `max_depth=8` — prevents memorising the training data
- `min_samples_leaf=12` — each decision rule needs at least 12 examples behind it
- `class_weight='balanced'` — all 5 classes treated equally

**How it works:** At each node, the tree picks the feature + threshold that best separates the classes. It measures "best" using Gini impurity: `Gini = 1 - Σ(pᵢ²)`. A pure node (all one class) has Gini = 0.


In [ ]:
clf = DecisionTreeClassifier(
    max_depth=8,
    min_samples_leaf=12,
    min_samples_split=20,
    class_weight='balanced',
    criterion='gini',
    random_state=42
)
clf.fit(X_train, y_train)

print(f"Leaves     : {clf.get_n_leaves()}")
print(f"Max depth  : {clf.get_depth()}")
print(f"Train acc  : {clf.score(X_train, y_train):.3f}")
print(f"Test acc   : {clf.score(X_test, y_test):.3f}")

## 10. Cross-Validation

We run the full train/test cycle 5 times on different splits. If the accuracy is consistent, the model generalises — it's not getting lucky.

In [ ]:
cv = cross_val_score(clf, X, y, cv=5, scoring='accuracy')
for i, s in enumerate(cv, 1):
    print(f"  Fold {i}: {s:.3f}")
print(f"  Mean:   {cv.mean():.3f}  StdDev: {cv.std():.3f}")

## 11. Classification Report

In [ ]:
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred, target_names=sorted(set(y))))

## 12. Confusion Matrix

In [ ]:
label_order = ['focused','skimming','confused','zoning_out','overloaded']
cm = confusion_matrix(y_test, y_pred, labels=label_order)
fig, ax = plt.subplots(figsize=(7,5))
ConfusionMatrixDisplay(cm, display_labels=label_order).plot(ax=ax, cmap='Blues', colorbar=True)
ax.set_title('Confusion Matrix — Test Set', fontweight='bold')
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.savefig('confusion_matrix.png', bbox_inches='tight', dpi=140)
plt.show()
print("confused vs overloaded share high-regression + low-scroll; separated by saccade_length and line_reread_count")

## 13. Feature Importance

Which features does the tree rely on most?

In [ ]:
imp = pd.Series(clf.feature_importances_, index=FEATURES).sort_values()
colors = ['#F44336' if v>0.15 else '#2196F3' if v>0.08 else '#90CAF9' for v in imp]
fig, ax = plt.subplots(figsize=(9,5))
bars = ax.barh(imp.index, imp.values, color=colors, height=0.55)
for bar, val in zip(bars, imp.values):
    ax.text(val+0.003, bar.get_y()+bar.get_height()/2, f'{val:.3f}', va='center', fontsize=9)
ax.set_xlabel('Importance (Gini reduction)'); ax.set_title('Feature Importance', fontweight='bold')
ax.grid(axis='x', alpha=0.25); ax.axvline(0.1, color='gray', ls='--', alpha=0.4)
plt.tight_layout()
plt.savefig('feature_importance.png', bbox_inches='tight', dpi=140)
plt.show()
print("Top feature:", imp.idxmax(), f"({imp.max():.3f})")

## 14. Decision Rules (top 4 levels)

These are the actual rules the model learned. Read them aloud in a pitch.

In [ ]:
print(export_text(clf, feature_names=FEATURES, max_depth=4, show_weights=True)[:2200])
print(f"... ({clf.get_n_leaves()} leaves total)")

## 15. Decision Tree Visualisation

In [ ]:
fig, ax = plt.subplots(figsize=(26,10))
plot_tree(clf, feature_names=FEATURES, class_names=clf.classes_,
          filled=True, rounded=True, fontsize=7, max_depth=4, ax=ax)
ax.set_title(f'Decision Tree (top 4 of {clf.get_depth()} levels)', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.savefig('decision_tree_visual.png', bbox_inches='tight', dpi=110)
plt.show()

## 16. Export as JavaScript

The tree is walked recursively and each decision node becomes a JavaScript `if/else`.
This is what runs in the browser extension — no ML library needed.


In [ ]:
def tree_to_js(tree, feature_names, class_names):
    tree_ = tree.tree_
    fn = [feature_names[i] if i != _tree.TREE_UNDEFINED else "?" for i in tree_.feature]
    lines = []
    def walk(node, depth):
        pad = '  ' * depth
        if tree_.feature[node] != _tree.TREE_UNDEFINED:
            lines.append(f"{pad}if (f.{fn[node]} <= {tree_.threshold[node]:.4f}) {{")
            walk(tree_.children_left[node], depth+1)
            lines.append(f"{pad}}} else {{")
            walk(tree_.children_right[node], depth+1)
            lines.append(f"{pad}}}")
        else:
            vals = tree_.value[node][0]
            idx  = vals.argmax()
            conf = vals[idx] / vals.sum()
            lines.append(f"{pad}return {{ label: '{class_names[idx]}', confidence: {conf:.3f} }};")
    walk(0, 1)
    return '\n'.join(lines)

body     = tree_to_js(clf, FEATURES, clf.classes_.tolist())
test_acc = clf.score(X_test, y_test)
n_train  = len(X_train)
n_leaves = clf.get_n_leaves()

js = (
    "// classifier.js - Decision Tree for TL;DR\n"
    "// Auto-generated from notebook - do not edit by hand\n"
    f"// Dataset: gaze_dataset.csv | Samples: {n_train} train | Test accuracy: {test_acc:.3f}\n"
    f"// Leaves: {n_leaves} | Features: {', '.join(FEATURES)}\n\n"
    "export function classifyGazeState(f) {\n"
    + body + "\n"
    "  return { label: 'focused', confidence: 0.5 };\n"
    "}\n\n"
    "export const COGNITIVE_STATE_ACTIONS = {\n"
    "  focused:    'none',\n"
    "  skimming:   'none',\n"
    "  confused:   'explain',\n"
    "  zoning_out: 'nudge',\n"
    "  overloaded: 'simplify',\n"
    "};\n"
)

with open('classifier.js', 'w') as f:
    f.write(js)

print(f"classifier.js written ({len(js.splitlines())} lines)")
print(f"Test accuracy: {test_acc:.3f}")

## 17. Live Test — Hand-crafted Examples

In [ ]:
ACTIONS = {
    'confused':   'Show AI explanation popup',
    'overloaded': 'Show simplified version',
    'zoning_out': 'Highlight paragraph (nudge)',
    'focused':    'Stay silent',
    'skimming':   'Stay silent',
}
examples = {
    'Confused reader':   [520, 140, 0.41,  38, 16, 21,   3, 130, 4.2],
    'Skimming reader':   [102,  35, 0.04, 240, 80, 30, 110, 600, 0.1],
    'Zoning out':        [1050, 230,0.15,  90, 70, 65,   1,  80, 0.3],
    'Focused reader':    [225,  52, 0.09,  95, 30, 14,  40, 290, 0.8],
    'Overloaded reader': [570, 165, 0.46,  30, 13, 16,   2, 118, 5.5],
}
for name, vals in examples.items():
    pred = clf.predict([vals])[0]
    conf = clf.predict_proba([vals])[0].max()
    print(f"{name:22s}  ->  {pred:12s}  ({conf*100:.0f}%)  |  {ACTIONS[pred]}")